# Middle East — Natural Resources Atlas

Country-by-country multilayer analysis, starting with **Saudi Arabia**.

### Data Sources
- **World Bank Indicators API v2** — resource rents (oil, gas, mineral, coal, forest) as % of GDP
- **Natural Earth 50m** — country boundary geometries
- **Curated geolocated dataset** — ~45 resource sites from Wikipedia, USGS, Ma'aden, Aramco public data

### Resource Categories
| Color | Type | Examples |
|-------|------|----------|
| 🟡 Gold | Oil Fields | Ghawar, Shaybah, Safaniyah, Khurais |
| 🔵 Steel Blue | Gas Fields | Karan, Hasbah, Arabiyah |
| 🟤 Copper | Mineral Mines | Mahd Ad Dhahab (gold), Jabal Sayid (copper), Al-Jalamid (phosphate) |
| ☀️ Yellow | Solar Farms | Sudair, Sakaka, NEOM |
| 💧 Sky Blue | Desalination | Ras Al-Khair, Jubail, Shuaiba |
| ⚙️ Gray | Refineries | Ras Tanura, Yanbu, Jubail |


In [1]:
# ── Cell 2: Fetch World Bank resource-rent indicators ─────────────────────────
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

COUNTRIES = {
    "SAU": "Saudi Arabia", "IRQ": "Iraq", "IRN": "Iran",
    "ARE": "UAE", "KWT": "Kuwait", "QAT": "Qatar",
    "OMN": "Oman", "BHR": "Bahrain",
}
COUNTRY_CODES = ";".join(COUNTRIES.keys())

INDICATORS = {
    "NY.GDP.PETR.RT.ZS": "Oil rents (% of GDP)",
    "NY.GDP.NGAS.RT.ZS": "Natural gas rents (% of GDP)",
    "NY.GDP.MINR.RT.ZS": "Mineral rents (% of GDP)",
    "NY.GDP.COAL.RT.ZS": "Coal rents (% of GDP)",
    "NY.GDP.FRST.RT.ZS": "Forest rents (% of GDP)",
    "NY.GDP.TOTL.RT.ZS": "Total resource rents (% of GDP)",
    "NY.GDP.MKTP.CD":    "GDP (current US$)",
    "SP.POP.TOTL":       "Population, total",
}

def fetch_wb(indicator, countries=COUNTRY_CODES, date="2000:2023"):
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator}"
    all_rows, page = [], 1
    while True:
        params = {"format": "json", "per_page": 500, "date": date, "page": page}
        r = requests.get(url, params=params, timeout=30)
        raw = r.json()
        if len(raw) < 2 or raw[1] is None:
            break
        for rec in raw[1]:
            if rec["value"] is not None:
                all_rows.append({
                    "country": rec["country"]["value"],
                    "iso3": rec["countryiso3code"],
                    "year": int(rec["date"]),
                    "value": rec["value"],
                })
        if page >= raw[0]["pages"]:
            break
        page += 1
    return all_rows

print("Fetching World Bank indicators …")
all_data = {}
for code, label in INDICATORS.items():
    rows = fetch_wb(code)
    all_data[code] = pd.DataFrame(rows)
    n = len(rows)
    status = "✓" if n > 0 else "✗ (no data)"
    print(f"  {status} {code:<25s} → {n:>4d} rows  │ {label}")

# ── Latest snapshot summary ──
def latest_value(df, iso3):
    if df.empty or "iso3" not in df.columns:
        return None, None
    sub = df[df["iso3"] == iso3].sort_values("year", ascending=False)
    if sub.empty:
        return None, None
    row = sub.iloc[0]
    return row["value"], int(row["year"])

summary_rows = []
for iso3, name in COUNTRIES.items():
    row = {"Country": name, "ISO3": iso3}
    for code, label in INDICATORS.items():
        df = all_data[code]
        val, yr = latest_value(df, iso3)
        key = f"{label} ({yr})" if val is not None else label
        row[key] = val
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Country")
print("\nAll indicators fetched ✓\n")
df_summary.T


Fetching World Bank indicators …
  ✓ NY.GDP.PETR.RT.ZS         →  175 rows  │ Oil rents (% of GDP)
  ✓ NY.GDP.NGAS.RT.ZS         →  175 rows  │ Natural gas rents (% of GDP)
  ✓ NY.GDP.MINR.RT.ZS         →  175 rows  │ Mineral rents (% of GDP)
  ✓ NY.GDP.COAL.RT.ZS         →  175 rows  │ Coal rents (% of GDP)
  ✓ NY.GDP.FRST.RT.ZS         →  175 rows  │ Forest rents (% of GDP)
  ✓ NY.GDP.TOTL.RT.ZS         →  175 rows  │ Total resource rents (% of GDP)
  ✓ NY.GDP.MKTP.CD            →  192 rows  │ GDP (current US$)
  ✓ SP.POP.TOTL               →  192 rows  │ Population, total

All indicators fetched ✓



Country,Saudi Arabia,Iraq,Iran,UAE,Kuwait,Qatar,Oman,Bahrain
ISO3,SAU,IRQ,IRN,ARE,KWT,QAT,OMN,BHR
Oil rents (% of GDP) (2021),23.686294,42.787098,18.265898,15.673095,NaN,15.278467,23.538744,10.937676
Natural gas rents (% of GDP) (2021),1.715848,0.655523,8.808886,1.960895,NaN,12.011255,5.67148,5.701911
Mineral rents (% of GDP) (2021),0.162202,0.0,3.35588,0.0,NaN,0.0,0.0,0.0
Coal rents (% of GDP) (2021),0.0,0.0,0.009395,0.0,NaN,0.0,0.0,0.0
Forest rents (% of GDP) (2021),0.000897,0.003236,0.008001,0.000106,NaN,0.000066,0.001355,0.000448
Total resource rents (% of GDP) (2021),25.565242,43.445858,30.44806,17.634096,NaN,27.289787,29.211579,16.640034
GDP (current US$) (2023),1218584533333.330078,268881051643.549011,457510482317.482971,522622191967.325012,165384407116.233002,217308516483.515991,106174708036.509995,46192260638.297897
"Population, total (2023)",33702731,45074049,90608707,10483751,4853420,2656032,5049269,1577059
Oil rents (% of GDP) (2020),NaN,NaN,NaN,NaN,27.581689,NaN,NaN,NaN


In [2]:
# ── Cell 3: Geolocated resource sites — Saudi Arabia ─────────────────────────
# Sources: Wikipedia, USGS, Ma'aden, Saudi Aramco public filings, Saudipedia

resource_sites = [
    # ── OIL FIELDS ──────────────────────────────────────────────────────────
    {"name": "Ghawar",        "lat": 25.43,  "lon": 49.62,  "type": "Oil", "subtype": "Onshore", "notes": "World's largest conventional oil field, ~280 km long"},
    {"name": "Shaybah",       "lat": 22.51,  "lon": 53.95,  "type": "Oil", "subtype": "Onshore", "notes": "Located in the Rub' al Khali (Empty Quarter)"},
    {"name": "Safaniyah",     "lat": 28.28,  "lon": 48.75,  "type": "Oil", "subtype": "Offshore", "notes": "World's largest offshore oil field"},
    {"name": "Khurais",       "lat": 25.07,  "lon": 48.20,  "type": "Oil", "subtype": "Onshore", "notes": "Khurais–Abu Jifan–Mazalij complex, ~150 km E of Riyadh"},
    {"name": "Abqaiq",        "lat": 25.93,  "lon": 49.67,  "type": "Oil", "subtype": "Onshore", "notes": "Major processing facility & field"},
    {"name": "Berri",         "lat": 27.00,  "lon": 49.75,  "type": "Oil", "subtype": "Offshore", "notes": "Coastal/offshore field, Eastern Province"},
    {"name": "Manifa",        "lat": 27.28,  "lon": 48.93,  "type": "Oil", "subtype": "Offshore", "notes": "Shallow-water mega-field in Persian Gulf"},
    {"name": "Zuluf",         "lat": 28.00,  "lon": 48.85,  "type": "Oil", "subtype": "Offshore", "notes": "Offshore field in northern Gulf waters"},
    {"name": "Marjan",        "lat": 28.25,  "lon": 49.25,  "type": "Oil", "subtype": "Offshore", "notes": "Offshore field, major expansion 2020s"},
    {"name": "Haradh",        "lat": 24.13,  "lon": 49.07,  "type": "Oil", "subtype": "Onshore", "notes": "Southern extension of Ghawar complex"},
    {"name": "Khursaniyah",   "lat": 27.08,  "lon": 49.00,  "type": "Oil", "subtype": "Onshore", "notes": "Light crude & gas processing"},
    {"name": "Qatif",         "lat": 26.58,  "lon": 50.00,  "type": "Oil", "subtype": "Onshore", "notes": "Eastern Province coastal field"},
    {"name": "Abu Sa'fah",    "lat": 27.25,  "lon": 50.17,  "type": "Oil", "subtype": "Offshore", "notes": "Shared production zone with Bahrain"},

    # ── GAS FIELDS ──────────────────────────────────────────────────────────
    {"name": "Karan",         "lat": 27.80,  "lon": 49.50,  "type": "Gas", "subtype": "Offshore", "notes": "First non-associated offshore gas field developed"},
    {"name": "Hasbah",        "lat": 27.50,  "lon": 49.80,  "type": "Gas", "subtype": "Offshore", "notes": "Part of Wasit Gas Program"},
    {"name": "Arabiyah",      "lat": 27.60,  "lon": 49.90,  "type": "Gas", "subtype": "Offshore", "notes": "Part of Wasit Gas Program"},
    {"name": "Midyan",        "lat": 28.35,  "lon": 36.50,  "type": "Gas", "subtype": "Onshore", "notes": "Northwest Saudi Arabia, Tabuk Province"},
    {"name": "Jafurah",       "lat": 24.50,  "lon": 49.80,  "type": "Gas", "subtype": "Onshore", "notes": "Massive unconventional gas basin, under development"},

    # ── MINERAL MINES ───────────────────────────────────────────────────────
    {"name": "Mahd Ad Dhahab",  "lat": 23.50,  "lon": 40.87,  "type": "Mineral", "subtype": "Gold", "notes": "Historic gold mine, Arabian Shield"},
    {"name": "Ad Duwayhi",      "lat": 22.75,  "lon": 42.25,  "type": "Mineral", "subtype": "Gold", "notes": "Ma'aden operated, high-grade gold"},
    {"name": "Bulghah",         "lat": 24.30,  "lon": 41.40,  "type": "Mineral", "subtype": "Gold", "notes": "Open-pit gold mine, Ma'aden"},
    {"name": "Al Sukhaybarat",  "lat": 25.45,  "lon": 41.50,  "type": "Mineral", "subtype": "Gold", "notes": "Underground gold mine"},
    {"name": "Al Amar",         "lat": 22.85,  "lon": 44.55,  "type": "Mineral", "subtype": "Gold", "notes": "Underground gold-copper mine"},
    {"name": "Jabal Sayid",     "lat": 23.85,  "lon": 40.94,  "type": "Mineral", "subtype": "Copper", "notes": "Major copper mine, Al Madinah Province"},
    {"name": "Al Masane",       "lat": 19.00,  "lon": 43.50,  "type": "Mineral", "subtype": "Copper/Zinc", "notes": "Copper-zinc-gold, Najran Province"},
    {"name": "Al-Jalamid",      "lat": 31.50,  "lon": 39.75,  "type": "Mineral", "subtype": "Phosphate", "notes": "Hazm Al-Jalamid, Ma'aden phosphate complex"},
    {"name": "Umm Wu'al",       "lat": 31.20,  "lon": 37.25,  "type": "Mineral", "subtype": "Phosphate", "notes": "Second phosphate project, Northern Border"},
    {"name": "Al Ba'itha",      "lat": 26.50,  "lon": 43.20,  "type": "Mineral", "subtype": "Bauxite", "notes": "Bauxite mine, Qassim Province"},

    # ── SOLAR FARMS ─────────────────────────────────────────────────────────
    {"name": "Sudair Solar PV", "lat": 25.98,  "lon": 45.52,  "type": "Solar", "subtype": "PV", "notes": "1.5 GW, Sudair Industrial City"},
    {"name": "Sakaka Solar",    "lat": 29.95,  "lon": 40.20,  "type": "Solar", "subtype": "PV", "notes": "300 MW, Al Jouf Province"},
    {"name": "NEOM Green H2",   "lat": 28.13,  "lon": 34.92,  "type": "Solar", "subtype": "PV + Wind", "notes": "NEOM mega-project, green hydrogen"},
    {"name": "Al Shuaibah Solar","lat": 21.30, "lon": 39.10,  "type": "Solar", "subtype": "PV", "notes": "2.6 GW, near Jeddah"},
    {"name": "Ar Rass Solar",   "lat": 25.87,  "lon": 43.50,  "type": "Solar", "subtype": "PV", "notes": "700 MW, Qassim Province"},

    # ── DESALINATION ────────────────────────────────────────────────────────
    {"name": "Ras Al-Khair",    "lat": 27.54,  "lon": 49.14,  "type": "Desalination", "subtype": "MSF + RO", "notes": "World's largest desalination plant"},
    {"name": "Jubail Desal",    "lat": 27.00,  "lon": 49.65,  "type": "Desalination", "subtype": "MSF", "notes": "Jubail Industrial City complex"},
    {"name": "Shuaiba Desal",   "lat": 20.67,  "lon": 39.53,  "type": "Desalination", "subtype": "MSF + RO", "notes": "Red Sea coast, south of Jeddah"},
    {"name": "Yanbu Desal",     "lat": 24.05,  "lon": 38.00,  "type": "Desalination", "subtype": "RO", "notes": "Red Sea coast desalination"},

    # ── REFINERIES ──────────────────────────────────────────────────────────
    {"name": "Ras Tanura Refinery", "lat": 26.69,  "lon": 50.10,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Major export terminal & refinery"},
    {"name": "Yanbu Refinery",      "lat": 23.92,  "lon": 38.28,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Red Sea coast refinery complex"},
    {"name": "Jubail Refinery",     "lat": 27.01,  "lon": 49.66,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "SATORP JV with Total"},
    {"name": "Riyadh Refinery",     "lat": 24.60,  "lon": 46.75,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "Inland refinery for domestic supply"},
    {"name": "Jazan Refinery",      "lat": 16.90,  "lon": 42.55,  "type": "Refinery", "subtype": "Oil Refinery", "notes": "400 kbd, newest Aramco refinery"},
    {"name": "Ras Al-Khair Smelter","lat": 27.55,  "lon": 49.15,  "type": "Refinery", "subtype": "Aluminium Smelter", "notes": "Ma'aden aluminium smelting complex"},
]

df_resources = pd.DataFrame(resource_sites)

# Summary
print(f"Total resource sites: {len(df_resources)}\n")
print(df_resources.groupby("type").size().to_string())
print(f"\nSubtypes:")
print(df_resources.groupby(["type", "subtype"]).size().to_string())


Total resource sites: 43

type
Desalination     4
Gas              5
Mineral         10
Oil             13
Refinery         6
Solar            5

Subtypes:
type          subtype          
Desalination  MSF                  1
              MSF + RO             2
              RO                   1
Gas           Offshore             3
              Onshore              2
Mineral       Bauxite              1
              Copper               1
              Copper/Zinc          1
              Gold                 5
              Phosphate            2
Oil           Offshore             6
              Onshore              7
Refinery      Aluminium Smelter    1
              Oil Refinery         5
Solar         PV                   4
              PV + Wind            1


In [3]:
# ── Cell 4: Interactive Resource Map (Folium — no API key needed) ─────────────
import folium
from folium.plugins import MarkerCluster
import geopandas as gpd
import json

# ── Load country boundary ──
NE_URL = "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip"
print("Downloading Natural Earth 50m boundaries …")
world = gpd.read_file(NE_URL)

ME_NAMES = ["Saudi Arabia", "Iraq", "Iran", "United Arab Emirates",
            "Kuwait", "Qatar", "Oman", "Bahrain", "Yemen", "Jordan",
            "Israel", "Syria", "Lebanon", "Turkey", "Egypt"]
gdf = world[world["ADMIN"].isin(ME_NAMES)][["ADMIN", "ISO_A3", "geometry"]].copy()
gdf = gdf.to_crs(epsg=4326)
print(f"Loaded {len(gdf)} country boundaries")

# ── Color map (hex) ──
COLOR_MAP = {
    "Oil":          "#DAA520",   # Gold/Amber
    "Gas":          "#4682B4",   # Steel Blue
    "Mineral":      "#B87333",   # Copper
    "Solar":        "#FFC800",   # Bright Yellow
    "Desalination": "#00BFFF",   # Deep Sky Blue
    "Refinery":     "#A9A9A9",   # Gray
}

ICON_MAP = {
    "Oil":          "tint",
    "Gas":          "fire",
    "Mineral":      "gem",
    "Solar":        "sun-o",
    "Desalination": "tint",
    "Refinery":     "industry",
}

# ── Build Folium map ──
m = folium.Map(
    location=[24.5, 45.0],
    zoom_start=5,
    tiles="CartoDB dark_matter",
    attr="CartoDB",
)

# Add country boundaries
saudi_gdf = gdf[gdf["ADMIN"] == "Saudi Arabia"]
neighbors_gdf = gdf[gdf["ADMIN"] != "Saudi Arabia"]

# Neighbors — subtle
folium.GeoJson(
    neighbors_gdf.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#333333",
        "color": "#555555",
        "weight": 1,
        "fillOpacity": 0.15,
    },
    name="Neighbors",
).add_to(m)

# Saudi Arabia — highlighted with gold border
folium.GeoJson(
    saudi_gdf.__geo_interface__,
    style_function=lambda x: {
        "fillColor": "#2d2823",
        "color": "#DAA520",
        "weight": 2.5,
        "fillOpacity": 0.25,
    },
    name="Saudi Arabia",
).add_to(m)

# ── Add resource markers ──
for _, row in df_resources.iterrows():
    color = COLOR_MAP.get(row["type"], "#FFFFFF")
    icon_name = ICON_MAP.get(row["type"], "info-sign")

    popup_html = (
        f"<div style='font-family:Inter,sans-serif;min-width:200px;'>"
        f"<b style='font-size:14px;color:{color};'>{row['name']}</b><br/>"
        f"<span style='color:#888;'>Type:</span> {row['type']} — {row['subtype']}<br/>"
        f"<span style='color:#888;'>Notes:</span> {row['notes']}"
        f"</div>"
    )

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=2,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['name']} ({row['type']})",
    ).add_to(m)

# ── Add legend ──
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:rgba(20,20,35,0.92); padding:14px 18px; border-radius:8px;
     border:1px solid #DAA520; font-family:Inter,sans-serif; color:#e0e0e0;
     font-size:12px; line-height:1.8; box-shadow:0 4px 16px rgba(0,0,0,0.5);">
  <b style="font-size:13px; color:#DAA520;">⬡ Resource Types</b><br/>
  <span style="color:#DAA520;">●</span> Oil Fields (13)<br/>
  <span style="color:#4682B4;">●</span> Gas Fields (5)<br/>
  <span style="color:#B87333;">●</span> Mineral Mines (10)<br/>
  <span style="color:#FFC800;">●</span> Solar Farms (5)<br/>
  <span style="color:#00BFFF;">●</span> Desalination (4)<br/>
  <span style="color:#A9A9A9;">●</span> Refineries (6)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Add layer control
folium.LayerControl().add_to(m)

# Save
output_path = "output/middle_east_resources.html"
m.save(output_path)
print(f"✓ Map saved to {output_path}")
print(f"  {len(df_resources)} resource sites plotted")
print(f"  {len(gdf)} country boundaries rendered")


Loaded 15 country boundaries
✓ Map saved to output/middle_east_resources.html
  43 resource sites plotted
  15 country boundaries rendered


In [4]:
# ── Cell 5: 3D Production Capacity Map (PyDeck ColumnLayer) ───────────────────
import pydeck as pdk
import pandas as pd
from pathlib import Path

BAR_ELEVATION_SCALE = 3000
LABEL_Z_OFFSET = 20000

# Add a normalized capacity scale (1-100) to represent relative production size.
# For example, Ghawar is massive (100), while smaller fields/plants get lower values.
capacity_mapping = {
    "Ghawar": 100, "Safaniyah": 80, "Khurais": 60, "Shaybah": 50, "Manifa": 50, "Zuluf": 45, "Marjan": 45,
    "Abqaiq": 40, "Berri": 35, "Haradh": 30, "Khursaniyah": 30, "Qatif": 25, "Abu Sa'fah": 25,
    "Karan": 40, "Hasbah": 35, "Arabiyah": 35, "Jafurah": 50, "Midyan": 15,
    "Mahd Ad Dhahab": 20, "Jabal Sayid": 25, "Al-Jalamid": 35, "Umm Wu'al": 30, "Al Ba'itha": 30,
    "Ad Duwayhi": 20, "Bulghah": 15, "Al Sukhaybarat": 15, "Al Amar": 15, "Al Masane": 15,
    "Sudair Solar PV": 35, "Sakaka Solar": 15, "NEOM Green H2": 45, "Al Shuaibah Solar": 40, "Ar Rass Solar": 25,
    "Ras Al-Khair": 50, "Jubail Desal": 40, "Shuaiba Desal": 35, "Yanbu Desal": 25,
    "Ras Tanura Refinery": 60, "Yanbu Refinery": 45, "Jubail Refinery": 45, "Jazan Refinery": 40,
    "Riyadh Refinery": 25, "Ras Al-Khair Smelter": 35
}

# Build a JSON-serializable copy for PyDeck. Passing the pandas DataFrame
# directly can serialize as an empty internal manager object in newer pandas.
df_3d = df_resources.copy()
df_3d['capacity_scale'] = df_3d['name'].map(capacity_mapping).fillna(10).astype(float)
df_3d['label_z'] = df_3d['capacity_scale'] * BAR_ELEVATION_SCALE + LABEL_Z_OFFSET
df_3d['label_text'] = df_3d.apply(lambda row: f"{row['name']}\nOutput: {row['capacity_scale']:.0f}/100", axis=1)

# Map colors to RGB lists for PyDeck
COLOR_MAP_RGB = {
    "Oil": [218, 165, 32, 220],     # Gold/Amber
    "Gas": [70, 130, 180, 220],     # Steel Blue
    "Mineral": [184, 115, 51, 220],     # Copper
    "Solar": [255, 200, 0, 220],      # Bright Yellow
    "Desalination": [0, 191, 255, 220],      # Deep Sky Blue
    "Refinery": [169, 169, 169, 220],    # Gray
}
df_3d['color_rgb'] = df_3d['type'].map(COLOR_MAP_RGB)
df_3d['color_rgb'] = df_3d['color_rgb'].apply(lambda c: c if isinstance(c, list) else [255, 255, 255, 220])
column_data = df_3d[
    ['name', 'lat', 'lon', 'type', 'subtype', 'notes', 'capacity_scale', 'label_z', 'label_text', 'color_rgb']
].to_dict('records')

print("Building 3D ColumnLayer...")

layer_3d = pdk.Layer(
    "ColumnLayer",
    column_data,
    get_position=["lon", "lat"],
    get_elevation="capacity_scale",
    elevation_scale=BAR_ELEVATION_SCALE,  # Adjust height multiplier
    radius=12000,          # Adjust column width
    get_fill_color="color_rgb",
    extruded=True,
    pickable=True,
    auto_highlight=True,
)

label_layer = pdk.Layer(
    "TextLayer",
    column_data,
    get_position=["lon", "lat", "label_z"],
    get_text="label_text",
    get_size=9,
    get_color=[245, 245, 245, 230],
    get_pixel_offset=[0, -6],
    billboard=True,
    pickable=False,
)

view_state_3d = pdk.ViewState(
    latitude=24.5,
    longitude=45.0,
    zoom=5,
    pitch=45,  # Tilted to see 3D effect
    bearing=0,
)

deck_3d = pdk.Deck(
    layers=[layer_3d, label_layer],
    initial_view_state=view_state_3d,
    tooltip={
        "html": "<b>{name}</b><br>" \
                "Type: {type} - {subtype}<br>" \
                "Relative Capacity: {capacity_scale}/100",
        "style": {"backgroundColor": "#1a1a2e", "color": "white", "border": "1px solid #daa520"}
    },
    map_style="dark" # Standard dark map
)

output_3d_path = "output/middle_east_3d_bars.html"
output_file = Path(output_3d_path)
deck_3d.to_html(output_3d_path)

def rgb_to_hex(rgb):
    return f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"

legend_items = "\n".join(
    f"<div style='display:flex;align-items:center;gap:8px;margin:3px 0;'>"
    f"<span style='width:10px;height:10px;border-radius:2px;display:inline-block;background:{rgb_to_hex(rgb)};'></span>"
    f"<span>{resource_type}</span>"
    f"</div>"
    for resource_type, rgb in COLOR_MAP_RGB.items()
)
legend_html = (
    "<div id='resource-legend' style='position:fixed;bottom:24px;left:24px;z-index:10;"
    "background:rgba(12,14,24,0.9);border:1px solid rgba(218,165,32,0.75);"
    "border-radius:8px;padding:10px 12px;color:#f3f4f8;font-family:Inter,Arial,sans-serif;"
    "font-size:11px;line-height:1.3;box-shadow:0 8px 24px rgba(0,0,0,0.35);'>"
    "<div style='font-weight:700;margin-bottom:6px;color:#ffffff;'>Resource type</div>"
    f"{legend_items}"
    "<div style='margin-top:8px;color:#c7cad4;font-size:10px;'>Bar height = relative production/capacity</div>"
    "</div>"
)
html = output_file.read_text(encoding="utf-8")
if "</body>" in html:
    html = html.replace("</body>", f"{legend_html}\n  </body>", 1)
else:
    html += legend_html
output_file.write_text(html, encoding="utf-8")
print(f"✓ 3D Map saved to {output_3d_path}")


Building 3D ColumnLayer...
✓ 3D Map saved to output/middle_east_3d_bars.html
